In [ ]:
import numpy as np
import seaborn as sns # optional
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import matplotlib.image as mpimg
from pathlib import Path
import numpy as np
from scipy import ndimage as ndi

In [ ]:
data_dir = Path("sentinel-data")

files_2017 = sorted((here / 'S2_2017-07-12').rglob('*.tiff'))
files_2018 = sorted((here / 'S2_2018-07-12').rglob('*.tiff'))
files_2019 = sorted((here / 'S2_2019-07-12').rglob('*.tiff'))

files_by_year = {"2017": files_2017, "2018": files_2018, "2019": files_2019}
for folder in sorted(data_dir.glob("S2_*")):
    print(folder.name)
    for f in sorted(folder.iterdir()):
        print("   ", f.name)

In [ ]:
file = files_2017[3]
with rasterio.open(file) as src:
    print("Resolution:", src.res)
    print("Coordinate reference system:", src.crs)
    print("Bounds:", src.bounds)
    print("Data type:", src.dtypes)
    print("NoData value:", src.nodata)
    print("Pixel area:", abs(src.transform.a * src.transform.e), "map units²")
    print("Transform:", src.transform)

Note that these are the same for all the files


* Natural Color: B4, B3, B2 (665nm, 560nm, 490nm respectively)
* Infrared: B8, B4, B3 (842nm, 665nm, 560nm)
* Short-Wave infrared: B12, B8A, B4 (2190nm, 865nm, 665nm)
* Agriculture: B11, B8, B2 (1610nm, 842nm, 490nm)
* Vegetation Index: (B8 - B4) / (B8 + B4)
* Red Edge Vegetation Index: (B3 - B4) / (B3 + B4) + (B6 - B4) / (B6 + B4)
* Moisture Index: (B8A - B11) / (B8A + B11)
* Normalized Water Index: (B3 - B8) / (B3 + B8)
* Ferrous Silicates: B12 / B11

In [ ]:
def pretty_stats(image_data):
    "print out stats with nice formatting"
    stats = {
        "Mean": np.mean(image_data),
        "Median": np.median(image_data),
        "Std Dev": np.std(image_data),
        "Min": np.min(image_data),
        "Max": np.max(image_data)
    }
    print("\nImage Statistics")
    print("-"*25)
    for label, value in stats.items():
        print(f"{label:<7} : {value:>15,.4f}")
    print("-"*25)

In [ ]:
with rasterio.open(files_2017[10]) as src:
    b12 = src.read(1)

pretty_stats(b12)

In [ ]:
# def make_ndi(band1, band2, band3=None):
#     """
#     take two bands and make a normalized difference image
#     """
#     band1[band1 <= 0] = np.nan
#     band2[band2 <= 0] = np.nan
#     ndi = (band1 - band2) / (band1 + band2)
#
#     if band3 is not None:
#         band3[band3 <= 0] = np.nan
#         ndi += (band3 - band2) / (band3 + band2)
#     return ndi

In [ ]:
def angular_pixel_calculation(file):
    with rasterio.open(file) as src:
        # Get image metadata
        resolution = src.res
        bounds = src.bounds

        width = src.width
        height = src.height

        # Calculate center latitude
        latitude = (bounds.bottom + bounds.top)/2

        # Convert angular resolution from degrees to metres
        pixel_width_m = (
            abs(resolution[0]) * 111320
            * np.cos(np.radians(latitude))
        )
        pixel_height_m = abs(resolution[1]) * 111320

        # Calculate pixel area and total image area
        pixel_area_m2 = pixel_width_m * pixel_height_m

        # Print results
        # print("Resolution:", resolution)
        # print("Image dimensions:", width, "x", height)
        # print("Total pixels:", total_pixels)
        # print("Bounds:", bounds)
        # print(f"Pixel width: {pixel_width_m:.2f} m")
        # print(f"Pixel height: {pixel_height_m:.2f} m")
        # print(f"Pixel area: {pixel_area_m2:.2f} m²")

        return pixel_area_m2

In [ ]:
for year, files in files_by_year.items():
    file = [f for f in files if "B12_Raw" in f.name][0]
    area = angular_pixel_calculation(file)
    side = np.sqrt(area)
    print(f"{year}: pixel area = {area:.2f} m², side = {side:.2f} m")

In [ ]:
def pixel_conversion(file, pixels):
    pixel_area_m2 = angular_pixel_calculation(file)
    return pixels * pixel_area_m2

In [ ]:
# Vegetation Index: (B8 - B4) / (B8 + B4)
def make_ndi(band1, band2, band3=None):
    """
    take two bands and make a normalized difference image
    """
    band1[band1 <= 0] = np.nan
    band2[band2 <= 0] = np.nan
    ndi = (band1 - band2) / (band1 + band2)

    if band3 is not None:
        band3[band3 <= 0] = np.nan
        ndi += (band3 - band2) / (band3 + band2)
    return ndi
#
# b8 = rasterio.open(files_2017[7]).read(1)
# b4 = rasterio.open(files_2017[3]).read(1)
# veggie = make_ndi(b8, b4)

In [ ]:
ndvi = {}
for year, files in files_by_year.items():
    b8 = rasterio.open(files[7]).read(1).astype(float)
    b4 = rasterio.open(files[3]).read(1).astype(float)
    ndvi[year] = make_ndi(b8, b4)

plt.close('all')
fig, axes = plt.subplots(1, 3, figsize=(18, 5), tight_layout=True)
for ax, (year, veggie) in zip(axes, ndvi.items()):
    im = ax.imshow(veggie, vmin=-1, vmax=1, cmap='inferno')
    ax.set_title(f"NDVI {year}")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()

In [ ]:
for year, files in files_by_year.items():
    print(year, Path(files[7]).name, Path(files[3]).name)

In [ ]:
 ndvi = {}

# Calculate NDVI for each year
for year, files in files_by_year.items():
    b8 = rasterio.open(files[7]).read(1).astype(float)
    b4 = rasterio.open(files[3]).read(1).astype(float)
    ndvi[year] = make_ndi(b8, b4)

# Count yellow-range pixels and convert to area
veg_counts = {}
veg_area = {}

for year, values in ndvi.items():
    count = np.sum((values >= 0.3) & (values <= 1))
    veg_counts[year] = count

    file = files_by_year[year][7]
    area = pixel_conversion(file, count)*1e-6
    veg_area[year] = area

    print(f"{year}: {count} pixels, {area:.2f} km²")

# Plot area over time
years = sorted(veg_area.keys())
areas = [veg_area[year] for year in years]

plt.plot(years, areas, marker="o")
plt.xlabel("Year")
plt.ylabel("Yellow-range area (m²)")
plt.title("NDVI Vegetation Over Time")
plt.grid(True)
plt.show()

In [ ]:
def vegetation_fraction(veg_counts, land_counts, plot=True):
    """Vegetation pixels / land pixels for each year. Returns {year: fraction}."""
    years = sorted(veg_counts)
    frac = {y: veg_counts[y] / land_counts[y] for y in years}

    for y in years:
        print(f"{y}: {veg_counts[y]} veg px / {land_counts[y]} land px = {frac[y]:.3f} ({100 * frac[y]:.1f}%)")

    if plot:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot([int(y) for y in years], [frac[y] for y in years], marker="o")
        ax.set_xticks([int(y) for y in years])
        ax.set_xlabel("Year")
        ax.set_ylabel("Vegetation / land area")
        ax.set_title("Vegetation as a fraction of land area")
        ax.grid(True)
        plt.show()

    return frac

veg_fraction = vegetation_fraction(yellow_counts, n_land)

## Compare with the given NVDI Files

In [ ]:
plt.close('all')
fig, axes = plt.subplots(1, len(files_by_year), figsize=(6 * len(files_by_year), 5), tight_layout=True)

pipeline_ndvi = {}
for ax, (year, files) in zip(axes, files_by_year.items()):
    f = [x for x in files if x.name.endswith("_NDVI.tiff")][0]
    with rasterio.open(f) as src:
        data = src.read(1).astype(float)
        r, g, b = src.read([1, 2, 3]).astype(float)

    #pipeline_ndvi[year] = data
    margin = 0.15
    data = ((g - r) > margin) & ((g - b) > margin)

    im = ax.imshow(data, cmap="inferno")
    ax.set_title(f"Pipeline NDVI {year}")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.show()

In [ ]:
def read_pipeline_rgb(files):
    f = [x for x in files if x.name.endswith("_NDVI.tiff")][0]
    with rasterio.open(f) as src:
        return np.moveaxis(src.read(), 0, -1).astype(float)      # (h, w, 3)

pipeline_rgb = {y: read_pipeline_rgb(f) for y, f in files_by_year.items()}
years = sorted(ndvi.keys())

plt.close('all')
fig, axes = plt.subplots(3, 3, figsize=(18, 16), tight_layout=True)

for i, year in enumerate(years):
    n = ndvi[year]
    rgb = pipeline_rgb[year]

    # Left: my NDVI
    im = axes[i, 0].imshow(n, vmin=-1, vmax=1, cmap="inferno")
    axes[i, 0].set_title(f"My NDVI {year}")
    axes[i, 0].axis("off")
    fig.colorbar(im, ax=axes[i, 0], fraction=0.046)

    # Middle: pipeline file shown as the RGB image it is (no vmin/vmax, no flipping)
    axes[i, 1].imshow(rgb)
    axes[i, 1].set_title(f"Pipeline file as RGB {year}")
    axes[i, 1].axis("off")

    # Right: how each colour channel varies with my NDVI (median per NDVI bin)
    ok = np.isfinite(n)
    bins = np.linspace(-1, 1, 81)
    centres = 0.5 * (bins[:-1] + bins[1:])
    idx = np.digitize(n[ok], bins) - 1
    for c, col in enumerate("rgb"):
        vals = rgb[..., c][ok]
        med = [np.median(vals[idx == b]) if np.sum(idx == b) > 20 else np.nan
               for b in range(len(centres))]
        axes[i, 2].plot(centres, med, color=col, label=col.upper())
    axes[i, 2].set_xlabel("My NDVI")
    axes[i, 2].set_ylabel("Pipeline channel value")
    axes[i, 2].set_title(f"Pipeline colour vs my NDVI {year}")
    axes[i, 2].legend()

plt.show()

In [ ]:
# plt.close('all')
# fig, axes = plt.subplots(1, 3, figsize=(18, 5), tight_layout=True)
#
# for ax, year in zip(axes, sorted(ndvi)):
#     diff = ndvi[year] - pipeline_ndvi[year]          # mine - pipeline, pixel by pixel
#     valid = np.isfinite(diff)
#
#     print(f"{year}: {valid.sum()} pixels compared, "
#           f"mean diff = {np.nanmean(diff):+.2e}, "
#           f"max |diff| = {np.nanmax(np.abs(diff)):.2e}, "
#           f"pixels with |diff| > 0.01: {(np.abs(diff[valid]) > 0.01).sum()}")
#
#     lim = max(np.nanmax(np.abs(diff)), 1e-6)
#     im = ax.imshow(diff, cmap="RdBu_r", vmin=-lim, vmax=lim)
#     ax.set_title(f"{year}: mine - pipeline")
#     ax.axis("off")
#     fig.colorbar(im, ax=ax, fraction=0.046, label="Delta NDVI")
#
# plt.show()

In [ ]:
# my_ndvi = {}
# for year, files in files_by_year.items():
#     b8 = rasterio.open(files[7]).read(1).astype(float)
#     b4 = rasterio.open(files[3]).read(1).astype(float)
#     my_ndvi[year] = make_ndi(b8, b4)
#
# fig, axes = plt.subplots(3, 3, figsize=(17, 13), tight_layout=True)
# rows = []
# for i, year in enumerate(sorted(my_ndvi)):
#     f = [x for x in files_by_year[year] if x.name.endswith("_NDVI.tiff")][0]
#     r, g, b = rasterio.open(f).read([1, 2, 3]).astype(float)
#
#     pipe = ((g - r) > 0.15) & ((g - b) > 0.15)
#     mine = my_ndvi[year] >= 0.12
#     diff = mine.astype(int) - pipe.astype(int)   # +1 only mine, -1 only pipeline
#
#     both = (mine & pipe).sum()
#     only_mine = (mine & ~pipe).sum()
#     only_pipe = (pipe & ~mine).sum()
#     rows.append({
#         "year": year,
#         "mine_px": mine.sum(),
#         "pipeline_px": pipe.sum(),
#         "both": both,
#         "only_mine": only_mine,
#         "only_pipeline": only_pipe,
#         "IoU": both / (both + only_mine + only_pipe),
#         "pixel_agreement_%": 100 * (diff == 0).mean(),
#     })
#
#     im0 = axes[i, 0].imshow(mine, cmap="gray", vmin=0, vmax=1)
#     axes[i, 0].set_title(f"Mine (NDVI >= 0.3) {year}")
#     fig.colorbar(im0, ax=axes[i, 0], fraction=0.046)
#
#     im1 = axes[i, 1].imshow(pipe, cmap="gray", vmin=0, vmax=1)
#     axes[i, 1].set_title(f"Pipeline {year}")
#     fig.colorbar(im1, ax=axes[i, 1], fraction=0.046)
#
#     im2 = axes[i, 2].imshow(diff, cmap="RdBu_r", vmin=-1, vmax=1)
#     axes[i, 2].set_title(f"Difference (mine - pipeline) {year}:")
#     fig.colorbar(im2, ax=axes[i, 2], fraction=0.046)
#
#     for a in axes[i]:
#         a.axis("off")
# plt.show()


In [ ]:
def compare_ndvi_masks(files_by_year, ndvi_thresh=0.12, margin=0.15, plot=True):
    """
    Compare my NDVI vegetation mask (NDVI >= ndvi_thresh) with the pipeline
    mask (green > red and blue by `margin`) for each year.
    Returns (my_ndvi, rows), where rows is a list of dicts (one per year).
    """
    years = sorted(files_by_year)
    my_ndvi, rows = {}, []

    if plot:
        fig, axes = plt.subplots(len(years), 3, figsize=(17, 4.3 * len(years)), tight_layout=True)
        axes = np.atleast_2d(axes)

    for i, year in enumerate(years):
        files = files_by_year[year]
        b8 = rasterio.open(files[7]).read(1).astype(float)
        b4 = rasterio.open(files[3]).read(1).astype(float)
        my_ndvi[year] = make_ndi(b8, b4)

        f = [x for x in files if x.name.endswith("_NDVI.tiff")][0]
        r, g, b = rasterio.open(f).read([1, 2, 3]).astype(float)

        pipe = ((g - r) > margin) & ((g - b) > margin)
        mine = my_ndvi[year] >= ndvi_thresh
        diff = mine.astype(int) - pipe.astype(int)   # +1 only mine, -1 only pipeline

        both = (mine & pipe).sum()
        only_mine = (mine & ~pipe).sum()
        only_pipe = (pipe & ~mine).sum()
        rows.append({
            "year": year,
            "mine_px": int(mine.sum()),
            "pipeline_px": int(pipe.sum()),
            "both": int(both),
            "only_mine": int(only_mine),
            "only_pipeline": int(only_pipe),
            "IoU": both / max(both + only_mine + only_pipe, 1),
            "agree_%": 100 * (diff == 0).mean(),
        })

        if plot:
            im0 = axes[i, 0].imshow(mine, cmap="gray", vmin=0, vmax=1)
            axes[i, 0].set_title(f"Mine (NDVI >= {ndvi_thresh}) {year}")
            fig.colorbar(im0, ax=axes[i, 0], fraction=0.046)

            im1 = axes[i, 1].imshow(pipe, cmap="gray", vmin=0, vmax=1)
            axes[i, 1].set_title(f"Pipeline {year}")
            fig.colorbar(im1, ax=axes[i, 1], fraction=0.046)

            im2 = axes[i, 2].imshow(diff, cmap="RdBu_r", vmin=-1, vmax=1)
            axes[i, 2].set_title(f"Difference (mine - pipeline) {year}: IoU = {rows[-1]['IoU']:.3f}")
            fig.colorbar(im2, ax=axes[i, 2], fraction=0.046)

            for a in axes[i]:
                a.axis("off")

    if plot:
        plt.show()

    # Plain text table
    print(f"{'Year':6s} {'mine_px':>9s} {'pipe_px':>9s} {'both':>9s} "
          f"{'only_mine':>10s} {'only_pipe':>10s} {'IoU':>7s} {'agree_%':>8s}")
    for r in rows:
        print(f"{r['year']:6s} {r['mine_px']:9d} {r['pipeline_px']:9d} {r['both']:9d} "
              f"{r['only_mine']:10d} {r['only_pipeline']:10d} {r['IoU']:7.3f} {r['agree_%']:8.2f}")

    return my_ndvi, rows

my_ndvi, stats = compare_ndvi_masks(files_by_year)

# To map as a ratio of land

In [ ]:
def load_b12(files):
    # look up by name
    f = [x for x in files if "B12_Raw" in x.name][0]
    with rasterio.open(f) as src:
        b12 = src.read(1).astype(float)
    b12[b12 <= 0] = np.nan                      # treat 0 / no-data as invalid
    return b12

def normalise_b12(b12):
    """Robust z-score relative to the ocean level of this image."""
    med = np.nanmedian(b12)
    mad = np.nanmedian(np.abs(b12 - med))
    sigma = 1.4826 * mad
    return (b12 - med) / sigma, med, sigma

b12_raw, b12_z = {}, {}
for year, files in files_by_year.items():
    b12_raw[year] = load_b12(files)
    b12_z[year], med, sigma = normalise_b12(b12_raw[year])
    print(f"{year}: ocean median = {med:.5f}, ocean sigma = {sigma:.5f}")

In [ ]:
# plt.close('all')
# fig, ax = plt.subplots(figsize=(8, 4))
# for year, z in b12_z.items():
#     h, e = np.histogram(z[np.isfinite(z)], bins=np.linspace(-5, 60, 300))
#     ax.plot(e[:-1], h, label=year)
# ax.semilogy()
# ax.set_xlabel("normalised B12 (z)"); ax.set_ylabel("pixels"); ax.legend()
# plt.show()

In [ ]:
def remove_small_objects(mask, min_size=500, connectivity=2, fill_holes=False):
    """
    Remove connected blobs of True pixels smaller than min_size (in pixels).

    connectivity: 1 = 4-neighbour (edges only), 2 = 8-neighbour (edges and diagonals)
    fill_holes:   also fill enclosed holes in the blobs that are kept
    """
    structure = ndi.generate_binary_structure(2, connectivity)
    lab, n = ndi.label(mask, structure=structure)
    if n == 0:
        return mask.copy()

    sizes = ndi.sum(mask, lab, index=np.arange(1, n + 1))
    keep_labels = np.where(sizes >= min_size)[0] + 1
    cleaned = np.isin(lab, keep_labels)

    if fill_holes:
        cleaned = ndi.binary_fill_holes(cleaned)
    return cleaned

In [ ]:
Z_THRESH = 5.0                      # same normalised threshold for every year
MIN_SIZE_BY_YEAR = {"2019": 4000}   # only these years get cloud removal {year: min_size}

# ---------- load and normalise B12 ----------
def load_b12(files):
    f = [x for x in files if "B12_Raw" in x.name][0]      # by name, not index
    with rasterio.open(f) as src:
        b12 = src.read(1).astype(float)
    b12[b12 <= 0] = np.nan                                # 0 / no-data -> invalid
    return b12

def normalise_b12(b12):
    """Robust z-score relative to this image's ocean level."""
    med = np.nanmedian(b12)
    sigma = 1.4826 * np.nanmedian(np.abs(b12 - med))
    return (b12 - med) / sigma, med, sigma

b12_raw, b12_z = {}, {}
for year, files in files_by_year.items():
    b12_raw[year] = load_b12(files)
    b12_z[year], med, sigma = normalise_b12(b12_raw[year])


land_masks, n_land = {}, {}
for year, z in b12_z.items():
    land = np.isfinite(z) & (z > Z_THRESH)
    n_before = int(land.sum())

    if year in MIN_SIZE_BY_YEAR:
        lab, n = ndi.label(land)
        if n:
            sizes = ndi.sum(land, lab, range(1, n + 1))
            land = np.isin(lab, 1 + np.where(sizes >= MIN_SIZE_BY_YEAR[year])[0])

    land_masks[year] = land
    n_land[year] = int(land.sum())
    note = f"  (blob filter min_size={MIN_SIZE_BY_YEAR[year]}, removed {n_before - n_land[year]} px)" \
           if year in MIN_SIZE_BY_YEAR else "  (no blob filter)"
    print(f"{year}: {n_land[year]} land px{note}")

# ---------- plot ----------
vmax = np.nanpercentile(b12_raw["2019"], 99.5)
plt.close('all')
fig, axes = plt.subplots(1, len(files_by_year), figsize=(6 * len(files_by_year), 5), tight_layout=True)
for ax, year in zip(axes, files_by_year):
    ax.imshow(np.ma.masked_array(b12_raw[year], mask=~land_masks[year]),
              vmin=0, vmax=vmax, cmap="viridis")
    ax.set_title(f"{year}: {n_land[year]} px")
    ax.axis("off")
plt.show()

In [ ]:
# land_area_m2 = {}
#
# for year, count in n_land.items():
#     # pick the file by name rather than files_by_year[year][7], since list order isn't guaranteed
#     file = [f for f in files_by_year[year] if "B12_Raw" in f.name][0]
#     land_area_m2[year] = pixel_conversion(file, count)
#
#     print(f"{year}: {count} pixels, {land_area_m2[year]:.0f} m², {land_area_m2[year] / 1e6:.3f} km²")
#
# # Plot land area over time
# years = sorted(land_area_m2.keys())
# areas_km2 = [land_area_m2[y] * 1e-6 for y in years]
#
# plt.close('all')
# fig, ax = plt.subplots(figsize=(6, 4))
# ax.plot([int(y) for y in years], areas_km2, marker="o")
# ax.set_xticks([int(y) for y in years])
# ax.set_xlabel("Year")
# ax.set_ylabel("Land area (km$^2$)")
# ax.set_title("Land area over time")
# ax.grid(True)
# plt.show()

In [ ]:
def land_area_over_time(n_land, files_by_year, plot=True):
    """Convert land pixel counts to km^2 for each year. Returns {year: km2}."""
    years = sorted(n_land)
    land_km2 = {}

    for y in years:
        # pick the file by name rather than by list position
        file = [f for f in files_by_year[y] if "B12_Raw" in f.name][0]
        land_km2[y] = pixel_conversion(file, n_land[y]) * 1e-6
        print(f"{y}: {n_land[y]} pixels, {land_km2[y] * 1e6:.0f} m², {land_km2[y]:.3f} km²")

    if plot:
        x = [int(y) for y in years]
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(x, [land_km2[y] for y in years], marker="o")
        ax.set_xticks(x)
        ax.set_xlabel("Year")
        ax.set_ylabel("Land area (km$^2$)")
        ax.set_title("Land area over time")
        ax.grid(True)
        plt.show()

    return land_km2

land_km2 = land_area_over_time(n_land, files_by_year)

In [ ]:
def vegetation_fraction(veg_counts, land_counts, files_by_year, plot=True):

    years = sorted(veg_counts)
    veg_km2 = {}
    land_km2 = {}
    frac = {}

    for y in years:
        file = [f for f in files_by_year[y] if "B12_Raw" in f.name][0]
        veg_km2[y] = pixel_conversion(file, veg_counts[y]) * 1e-6
        land_km2[y] = pixel_conversion(file, land_counts[y]) * 1e-6
        frac[y] = veg_km2[y] / land_km2[y]

        print(f"{y}: vegetation = {veg_km2[y]:.3f} km², land = {land_km2[y]:.3f} km², "
              f"fraction = {frac[y]:.3f} ({100 * frac[y]:.1f}%)")

    if plot:
        x = [int(y) for y in years]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), tight_layout=True)

        ax1.plot(x, [land_km2[y] for y in years], marker="o", label="Land")
        ax1.plot(x, [veg_km2[y] for y in years], marker="o", label="Vegetation")
        ax1.set_ylabel("Area (km$^2$)")
        ax1.set_title("Land and vegetation area")
        ax1.legend()

        ax2.plot(x, [frac[y] for y in years], marker="o", color="g")
        ax2.set_ylabel("Vegetation / land area")
        ax2.set_title("Vegetation as a fraction of land area")

        for ax in (ax1, ax2):
            ax.set_xticks(x)
            ax.set_xlabel("Year")
            ax.grid(True)
        plt.show()

    return veg_km2, land_km2, frac

veg_km2, land_km2, veg_fraction = vegetation_fraction(veg_counts, n_land, files_by_year)